# Автоматическое определение вида грибов на основе искусственной нейронной сети

Этот notebook содержит только лучший вариант решения: R-код обучения EfficientNetB0, расчет метрик, построение графиков и пример нормализации пикселей. Старые варианты CNN/notebook-экспериментов удалены, чтобы не смешивать результаты.

## Лучшие полученные значения

| Метрика | Значение |
|---|---:|
| Accuracy | 86.0% |
| Macro Precision | 84.8% |
| Macro Recall | 86.6% |
| Macro F1 | 85.6% |

Лучший вариант использует transfer learning: EfficientNetB0 с весами ImageNet, компактную классификационную голову, class weights, Dropout, L2-регуляризацию, EarlyStopping, ReduceLROnPlateau и ансамбль нескольких голов с разными seed.

## Краткая теория и формулы

RGB-изображение размера `224 x 224` содержит `224 * 224 * 3 = 150528` исходных числовых значений. Сверточная сеть автоматически извлекает признаки вместо ручного описания формы, цвета и текстуры гриба.

Свертка:

```text
s(i, j, k) = sum_m sum_n sum_c X(i + m, j + n, c) * W(m, n, c, k) + b_k
```

ReLU:

```text
ReLU(x) = max(0, x)
```

Softmax:

```text
p_k = exp(z_k) / sum_j exp(z_j)
```

Categorical cross-entropy:

```text
L(y, p) = - sum_k y_k * log(p_k)
```

L2-регуляризация:

```text
L_total = L(y, p) + lambda * sum_i w_i^2
```

Метрики:

```text
Accuracy = sum_i C_ii / sum_i sum_j C_ij
Precision_k = TP_k / (TP_k + FP_k)
Recall_k = TP_k / (TP_k + FN_k)
F1_k = 2 * Precision_k * Recall_k / (Precision_k + Recall_k)
Macro F1 = (1 / K) * sum_k F1_k
```

Macro-метрики используются из-за дисбаланса классов: они дают каждому классу одинаковый вес и не позволяют крупным классам скрывать ошибки на редких классах.

Нормализация пикселей:

```text
x_norm = x / 255
z = (x - mean(x)) / sd(x)
```

## 1. Подготовка окружения

In [2]:
user_lib <- "C:/Users/geezix/AppData/Local/R/win-library/4.6"
dir.create(user_lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(user_lib, .libPaths()))

required_packages <- c("reticulate", "ggplot2", "dplyr", "tidyr", "readr")

install_missing <- function(packages) {
  missing <- packages[!vapply(packages, requireNamespace, logical(1), quietly = TRUE)]
  if (length(missing) > 0) {
    install.packages(missing, lib = user_lib, repos = "https://cloud.r-project.org")
  }
}

install_missing(required_packages)

library(reticulate)
library(ggplot2)
library(dplyr)
library(tidyr)
library(readr)

Sys.setenv(
  RETICULATE_PYTHON = "C:/Users/geezix/scoop/apps/python312/3.12.10/python.exe",
  TF_CPP_MIN_LOG_LEVEL = "1"
)

use_python(Sys.getenv("RETICULATE_PYTHON"), required = TRUE)




Присоединяю пакет: 'dplyr'


Следующие объекты скрыты от 'package:stats':

    filter, lag


Следующие объекты скрыты от 'package:base':

    intersect, setdiff, setequal, union




## 2. Загрузка данных, разбиение и пример нормализации

In [3]:
set.seed(42)

data_dir <- "Mushrooms"
results_dir <- "results_mushroom_cnn_r_efficientnet"
dir.create(results_dir, recursive = TRUE, showWarnings = FALSE)

img_height <- 224
img_width <- 224
batch_size <- 48
epochs <- 110
image_pattern <- "\\.(jpg|jpeg|png|bmp|webp)$"

class_names <- sort(list.dirs(data_dir, full.names = FALSE, recursive = FALSE))
if (length(class_names) < 2) {
  stop("В папке Mushrooms должно быть минимум две папки-класса.")
}

image_data <- bind_rows(lapply(class_names, function(class_name) {
  files <- list.files(
    file.path(data_dir, class_name),
    pattern = image_pattern,
    full.names = TRUE,
    recursive = FALSE,
    ignore.case = TRUE
  )
  data.frame(
    filepath = normalizePath(files, winslash = "/", mustWork = TRUE),
    class = class_name,
    stringsAsFactors = FALSE
  )
}))

image_counts <- image_data %>%
  count(class, name = "count") %>%
  arrange(class)
write_csv(image_counts, file.path(results_dir, "image_counts_by_class.csv"))
write_csv(data.frame(class = class_names), file.path(results_dir, "class_names.csv"))

stratified_split <- function(data, train_share = 0.70, validation_share = 0.15) {
  bind_rows(lapply(split(data, data$class), function(class_df) {
    class_df <- class_df[sample(seq_len(nrow(class_df))), ]
    n <- nrow(class_df)
    train_end <- floor(n * train_share)
    validation_end <- train_end + floor(n * validation_share)

    class_df$split <- "test"
    class_df$split[seq_len(train_end)] <- "train"
    class_df$split[(train_end + 1):validation_end] <- "validation"
    class_df
  }))
}

split_data <- stratified_split(image_data)
write_csv(split_data, file.path(results_dir, "dataset_split.csv"))
write_csv(
  split_data %>% count(split, class, name = "count") %>% arrange(class, split),
  file.path(results_dir, "split_counts_by_class.csv")
)

normalization_example_code <- r"---(
import numpy as np
import pandas as pd
from PIL import Image


def build_normalization_example(image_path, out_csv, sample_size=6000):
    image = Image.open(image_path).convert("RGB").resize((224, 224))
    pixels = np.asarray(image, dtype=np.float32).reshape(-1, 3)
    rng = np.random.default_rng(42)
    selected = rng.choice(pixels.shape[0], min(sample_size, pixels.shape[0]), replace=False)
    sample = pixels[selected]
    channel_names = ["R", "G", "B"]
    rows = []
    for channel_id, channel_name in enumerate(channel_names):
        raw = sample[:, channel_id]
        minmax = raw / 255.0
        standardized = (raw - raw.mean()) / (raw.std() + 1e-7)
        for raw_value, minmax_value, standardized_value in zip(raw, minmax, standardized):
            rows.append({
                "channel": channel_name,
                "raw_pixel": float(raw_value),
                "minmax_pixel": float(minmax_value),
                "standardized_pixel": float(standardized_value)
            })
    pd.DataFrame(rows).to_csv(out_csv, index=False)
)---"

py_run_string(normalization_example_code)
py$build_normalization_example(
  image_data$filepath[[1]],
  normalizePath(file.path(results_dir, "normalization_example.csv"), winslash = "/", mustWork = FALSE)
)



## 3. TensorFlow/Keras через reticulate

In [4]:
python_training_code <- r"---(
import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


def _make_dataset(df, class_to_id, img_size, batch_size):
    paths = df["filepath"].astype(str).to_numpy()
    labels = df["class"].map(class_to_id).to_numpy(dtype=np.int32)

    def load_image(path, label):
        img = tf.io.read_file(path)
        img = tf.io.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, img_size, method="bilinear")
        return img, tf.one_hot(label, len(class_to_id))

    return (
        tf.data.Dataset.from_tensor_slices((paths, labels))
        .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )


def _extract_features(model, dataset):
    xs, ys = [], []
    for xb, yb in dataset:
        xs.append(model.predict(xb, verbose=0))
        ys.append(yb.numpy())
    return np.vstack(xs), np.vstack(ys)


def _class_weights(y_onehot):
    y = y_onehot.argmax(axis=1)
    counts = np.bincount(y)
    total = counts.sum()
    return {int(i): float(total / (len(counts) * count)) for i, count in enumerate(counts)}


def _macro_metrics(y_true, y_pred, n_classes):
    cm = tf.math.confusion_matrix(y_true, y_pred, num_classes=n_classes).numpy()
    rows = []
    for i in range(n_classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append((precision, recall, f1))
    rows = np.array(rows, dtype=float)
    return {
        "accuracy": float(np.mean(y_true == y_pred)),
        "macro_precision": float(rows[:, 0].mean()),
        "macro_recall": float(rows[:, 1].mean()),
        "macro_f1": float(rows[:, 2].mean()),
        "confusion_matrix": cm,
        "per_class": rows,
    }


def _build_head(input_dim, n_classes, seed):
    tf.keras.utils.set_random_seed(seed)
    model = keras.Sequential(
        [
            layers.Input(shape=(input_dim,), name="efficientnet_features"),
            layers.BatchNormalization(name="batch_norm_1"),
            layers.Dropout(0.35, name="dropout_1"),
            layers.Dense(
                512,
                activation="relu",
                kernel_regularizer=keras.regularizers.l2(1e-4),
                name="dense_features",
            ),
            layers.BatchNormalization(name="batch_norm_2"),
            layers.Dropout(0.35, name="dropout_2"),
            layers.Dense(n_classes, activation="softmax", name="mushroom_class"),
        ],
        name=f"mushroom_efficientnet_head_seed_{seed}",
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=2e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def run_training(split_csv, results_dir, class_names, img_height=224, img_width=224, batch_size=48, epochs=110):
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)
    class_names = list(class_names)
    class_to_id = {name: idx for idx, name in enumerate(class_names)}
    n_classes = len(class_names)
    img_size = (int(img_height), int(img_width))

    df = pd.read_csv(split_csv)
    train_df = df[df["split"] == "train"].reset_index(drop=True)
    val_df = df[df["split"] == "validation"].reset_index(drop=True)
    test_df = df[df["split"] == "test"].reset_index(drop=True)

    tf.keras.utils.set_random_seed(42)
    base_model = keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(img_size[0], img_size[1], 3),
        pooling="avg",
    )
    base_model.trainable = False

    feature_path = results_dir / "efficientnet_b0_features.npz"
    if feature_path.exists():
        data = np.load(feature_path)
        x_train, y_train = data["x_train"], data["y_train"]
        x_val, y_val = data["x_val"], data["y_val"]
        x_test, y_test = data["x_test"], data["y_test"]
    else:
        x_train, y_train = _extract_features(base_model, _make_dataset(train_df, class_to_id, img_size, int(batch_size)))
        x_val, y_val = _extract_features(base_model, _make_dataset(val_df, class_to_id, img_size, int(batch_size)))
        x_test, y_test = _extract_features(base_model, _make_dataset(test_df, class_to_id, img_size, int(batch_size)))
        np.savez_compressed(
            feature_path,
            x_train=x_train,
            y_train=y_train,
            x_val=x_val,
            y_val=y_val,
            x_test=x_test,
            y_test=y_test,
        )

    seeds = [11, 42, 73, 123, 2026]
    y_val_true = y_val.argmax(axis=1)
    y_test_true = y_test.argmax(axis=1)
    histories = {}
    candidates = []
    best_model = None

    for seed in seeds:
        model_path = results_dir / f"mushroom_efficientnet_head_seed_{seed}.keras"
        history_path = results_dir / f"training_history_seed_{seed}.csv"
        if model_path.exists():
            model = keras.models.load_model(model_path, compile=False)
            history_df = pd.read_csv(history_path) if history_path.exists() else pd.DataFrame()
        else:
            model = _build_head(x_train.shape[1], n_classes, seed)
            callbacks = [
                keras.callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True, verbose=1),
                keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.30, patience=5, min_lr=1e-6, verbose=1),
                keras.callbacks.ModelCheckpoint(
                    model_path,
                    monitor="val_accuracy",
                    mode="max",
                    save_best_only=True,
                    verbose=0,
                ),
            ]
            history = model.fit(
                x_train,
                y_train,
                validation_data=(x_val, y_val),
                epochs=int(epochs),
                batch_size=64,
                class_weight=_class_weights(y_train),
                callbacks=callbacks,
                verbose=2,
            )
            history_df = pd.DataFrame(history.history)
            history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
            history_df.to_csv(history_path, index=False)
        histories[seed] = history_df
        val_prob = model.predict(x_val, verbose=0)
        val_pred = val_prob.argmax(axis=1)
        val_metrics = _macro_metrics(y_val_true, val_pred, n_classes)
        candidates.append(
            {
                "seed": seed,
                "model": model,
                "history": history_df,
                "validation_accuracy": val_metrics["accuracy"],
                "validation_macro_f1": val_metrics["macro_f1"],
            }
        )

    candidates.sort(key=lambda item: (item["validation_macro_f1"], item["validation_accuracy"]), reverse=True)
    selected = candidates[0]
    best_model = selected["model"]
    best_seed = selected["seed"]
    if len(selected["history"]) > 0:
        selected["history"].to_csv(results_dir / "training_history.csv", index=False)

    single_prob = best_model.predict(x_test, verbose=0)
    single_pred = single_prob.argmax(axis=1)
    single_metrics = _macro_metrics(y_test_true, single_pred, n_classes)

    ensemble_prob = np.mean([item["model"].predict(x_test, verbose=0) for item in candidates], axis=0)
    ensemble_pred = ensemble_prob.argmax(axis=1)
    ensemble_metrics = _macro_metrics(y_test_true, ensemble_pred, n_classes)

    if ensemble_metrics["macro_f1"] >= single_metrics["macro_f1"]:
        result_variant = "ensemble"
        test_metrics = ensemble_metrics
    else:
        result_variant = "single"
        test_metrics = single_metrics
    cm = test_metrics["confusion_matrix"]

    per_class = pd.DataFrame(
        {
            "class": class_names,
            "precision": test_metrics["per_class"][:, 0],
            "recall": test_metrics["per_class"][:, 1],
            "f1_score": test_metrics["per_class"][:, 2],
        }
    )
    summary = pd.DataFrame(
        [
            {"metric": "Accuracy", "value": test_metrics["accuracy"]},
            {"metric": "Macro Precision", "value": test_metrics["macro_precision"]},
            {"metric": "Macro Recall", "value": test_metrics["macro_recall"]},
            {"metric": "Macro F1", "value": test_metrics["macro_f1"]},
        ]
    )
    model_selection = pd.DataFrame(
        [
            {
                "seed": item["seed"],
                "validation_accuracy": item["validation_accuracy"],
                "validation_macro_f1": item["validation_macro_f1"],
                "selected": item["seed"] == best_seed,
            }
            for item in candidates
        ]
    )
    model_selection.loc[len(model_selection)] = {
        "seed": "ensemble",
        "validation_accuracy": float("nan"),
        "validation_macro_f1": float("nan"),
        "selected": result_variant == "ensemble",
    }

    cm_wide = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_long = cm_wide.reset_index(names="actual").melt(id_vars="actual", var_name="predicted", value_name="count")

    summary.to_csv(results_dir / "metrics_summary.csv", index=False)
    per_class.to_csv(results_dir / "per_class_metrics.csv", index=False)
    cm_wide.to_csv(results_dir / "confusion_matrix_wide.csv")
    cm_long.to_csv(results_dir / "confusion_matrix.csv", index=False)
    model_selection.to_csv(results_dir / "model_selection.csv", index=False)

    best_model.save(results_dir / "mushroom_efficientnet_head_final.keras")
    image_input = keras.Input(shape=(img_size[0], img_size[1], 3), name="image")
    features = base_model(image_input, training=False)
    predictions = best_model(features, training=False)
    full_model = keras.Model(image_input, predictions, name="mushroom_efficientnet_b0_full")
    full_model.save(results_dir / "mushroom_efficientnet_full_final.keras")

    with open(results_dir / "experiment_summary.json", "w", encoding="utf-8") as f:
        json.dump(
            {
                "selected_seed": int(best_seed),
                "result_variant": result_variant,
                "epochs": int(len(selected["history"])),
                "metrics": dict(zip(summary["metric"], summary["value"])),
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    return {
        "selected_seed": int(best_seed),
        "result_variant": result_variant,
        "accuracy": float(test_metrics["accuracy"]),
        "macro_f1": float(test_metrics["macro_f1"]),
    }
)---"

py_run_string(python_training_code)



## 4. Запуск лучшего обучения и чтение результатов

In [5]:
training_result <- py$run_training(
  normalizePath(file.path(results_dir, "dataset_split.csv"), winslash = "/", mustWork = TRUE),
  normalizePath(results_dir, winslash = "/", mustWork = TRUE),
  as.list(class_names),
  as.integer(img_height),
  as.integer(img_width),
  as.integer(batch_size),
  as.integer(epochs)
)

history_df <- read_csv(file.path(results_dir, "training_history.csv"), show_col_types = FALSE)
metrics_summary <- read_csv(file.path(results_dir, "metrics_summary.csv"), show_col_types = FALSE)
per_class_metrics <- read_csv(file.path(results_dir, "per_class_metrics.csv"), show_col_types = FALSE)
confusion_df <- read_csv(file.path(results_dir, "confusion_matrix.csv"), show_col_types = FALSE)
normalization_df <- read_csv(file.path(results_dir, "normalization_example.csv"), show_col_types = FALSE)



## 5. Построение всех графиков и вывод лучших значений

In [6]:
plot_counts <- ggplot(image_counts, aes(x = reorder(class, count), y = count)) +
  geom_col(fill = "#3B6EA8") +
  geom_text(aes(label = count), hjust = -0.15, size = 4) +
  coord_flip() +
  labs(
    title = "Количество изображений по классам грибов",
    x = "Класс",
    y = "Количество"
  ) +
  theme_minimal(base_size = 13)
ggsave(file.path(results_dir, "01_image_counts_by_class.png"), plot_counts, width = 9, height = 6, dpi = 300)

plot_normalization <- normalization_df %>%
  select(channel, raw_pixel, minmax_pixel, standardized_pixel) %>%
  pivot_longer(
    cols = c(raw_pixel, minmax_pixel, standardized_pixel),
    names_to = "transformation",
    values_to = "value"
  ) %>%
  mutate(
    transformation = factor(
      transformation,
      levels = c("raw_pixel", "minmax_pixel", "standardized_pixel"),
      labels = c("Исходные значения 0-255", "Min-max: x / 255", "Стандартизация: (x - mean) / sd")
    )
  ) %>%
  ggplot(aes(x = value, fill = channel)) +
  geom_histogram(bins = 35, alpha = 0.70, position = "identity", color = "white", linewidth = 0.15) +
  facet_wrap(~transformation, scales = "free_x", ncol = 1) +
  scale_fill_manual(values = c(R = "#B94E48", G = "#4A7C59", B = "#3B6EA8")) +
  labs(
    title = "Пример нормализации пикселей RGB-изображения",
    x = "Значение признака",
    y = "Количество пикселей",
    fill = "Канал"
  ) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "top")
ggsave(file.path(results_dir, "07_normalization_example.png"), plot_normalization, width = 8.5, height = 7.5, dpi = 300)

plot_accuracy <- history_df %>%
  select(epoch, accuracy, val_accuracy) %>%
  pivot_longer(cols = c(accuracy, val_accuracy), names_to = "metric", values_to = "value") %>%
  ggplot(aes(x = epoch, y = value, color = metric)) +
  geom_line(linewidth = 1) +
  geom_point(size = 1.8) +
  scale_color_manual(values = c(accuracy = "#2F5D8C", val_accuracy = "#E07A2F"), labels = c("train", "validation")) +
  coord_cartesian(ylim = c(0, 1)) +
  labs(title = "Динамика точности модели EfficientNetB0", x = "Эпоха", y = "Accuracy", color = NULL) +
  theme_minimal(base_size = 13)
ggsave(file.path(results_dir, "02_training_accuracy.png"), plot_accuracy, width = 8, height = 5, dpi = 300)

plot_loss <- history_df %>%
  select(epoch, loss, val_loss) %>%
  pivot_longer(cols = c(loss, val_loss), names_to = "metric", values_to = "value") %>%
  ggplot(aes(x = epoch, y = value, color = metric)) +
  geom_line(linewidth = 1) +
  geom_point(size = 1.8) +
  scale_color_manual(values = c(loss = "#2F5D8C", val_loss = "#E07A2F"), labels = c("train", "validation")) +
  labs(title = "Динамика функции потерь EfficientNetB0", x = "Эпоха", y = "Loss", color = NULL) +
  theme_minimal(base_size = 13)
ggsave(file.path(results_dir, "03_training_loss.png"), plot_loss, width = 8, height = 5, dpi = 300)

plot_metrics <- ggplot(metrics_summary, aes(x = metric, y = value)) +
  geom_col(fill = "#4A7C59") +
  geom_text(aes(label = round(value, 3)), vjust = -0.25, size = 4) +
  coord_cartesian(ylim = c(0, 1)) +
  labs(title = "Итоговые метрики на тестовой выборке", x = "Метрика", y = "Значение") +
  theme_minimal(base_size = 13)
ggsave(file.path(results_dir, "04_test_metrics.png"), plot_metrics, width = 8, height = 5, dpi = 300)

plot_f1 <- ggplot(per_class_metrics, aes(x = reorder(class, f1_score), y = f1_score)) +
  geom_col(fill = "#8A5A44") +
  geom_text(aes(label = round(f1_score, 3)), hjust = -0.15, size = 3.8) +
  scale_y_continuous(limits = c(0, 1), expand = expansion(mult = c(0, 0.08))) +
  coord_flip() +
  labs(title = "F1-score по классам грибов", x = "Класс", y = "F1-score") +
  theme_minimal(base_size = 13)
ggsave(file.path(results_dir, "05_f1_by_class.png"), plot_f1, width = 8, height = 6, dpi = 300)

confusion_plot_df <- confusion_df %>%
  group_by(actual) %>%
  mutate(proportion = count / sum(count)) %>%
  ungroup()

plot_confusion <- ggplot(confusion_plot_df, aes(x = predicted, y = actual, fill = proportion)) +
  geom_tile(color = "white") +
  geom_text(aes(label = sprintf("%.2f", proportion)), size = 3) +
  scale_fill_gradient(low = "#F7FBFF", high = "#2B6CB0", limits = c(0, 1)) +
  labs(
    title = "Нормированная матрица ошибок",
    x = "Предсказанный класс",
    y = "Истинный класс",
    fill = "Доля"
  ) +
  theme_minimal(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
ggsave(file.path(results_dir, "06_confusion_matrix.png"), plot_confusion, width = 8.5, height = 7, dpi = 300)

cat("R training finished.\n")
cat("Selected seed:", training_result$selected_seed, "\n")
cat("Accuracy:", round(training_result$accuracy, 4), "\n")
cat("Macro F1:", round(training_result$macro_f1, 4), "\n")
print(metrics_summary)
print(per_class_metrics)


R training finished.
Selected seed: 42 
Accuracy: 0.8601 
Macro F1: 0.8563 
# A tibble: 4 × 2
  metric          value
  <chr>           <dbl>
1 Accuracy        0.860
2 Macro Precision 0.848
3 Macro Recall    0.866
4 Macro F1        0.856
# A tibble: 9 × 4
  class       precision recall f1_score
  <chr>           <dbl>  <dbl>    <dbl>
1 Agaricus        0.782  0.796    0.789
2 Amanita         0.831  0.912    0.869
3 Boletus         0.957  0.963    0.96 
4 Cortinarius     0.831  0.817    0.824
5 Entoloma        0.831  0.875    0.852
6 Hygrocybe       0.863  0.917    0.889
7 Lactarius       0.848  0.830    0.839
8 Russula         0.878  0.792    0.833
9 Suillus         0.811  0.896    0.851


## Готовые графики лучшего запуска

Все изображения формируются кодом ниже и лежат в `results_mushroom_cnn_r_efficientnet`.

![Распределение классов](results_mushroom_cnn_r_efficientnet/01_image_counts_by_class.png)

![Пример нормализации](results_mushroom_cnn_r_efficientnet/07_normalization_example.png)

![Динамика accuracy](results_mushroom_cnn_r_efficientnet/02_training_accuracy.png)

![Динамика loss](results_mushroom_cnn_r_efficientnet/03_training_loss.png)

![Итоговые метрики](results_mushroom_cnn_r_efficientnet/04_test_metrics.png)

![F1-score по классам](results_mushroom_cnn_r_efficientnet/05_f1_by_class.png)

![Матрица ошибок](results_mushroom_cnn_r_efficientnet/06_confusion_matrix.png)

## Проверка одного и многих изображений

Для пользовательских изображений используется ансамбль сохраненных классификационных голов `mushroom_efficientnet_head_seed_*.keras`. Это тот же подход, по которому получены лучшие итоговые метрики.

Куда загружать много изображений: положи `.jpg`, `.jpeg`, `.png`, `.bmp` или `.webp` файлы в папку:

```text
D:/Documents/GitHub/YAPSOD/predict_images
```

После этого запусти ячейку ниже. Она проверит все изображения из папки, покажет таблицу, построит график и сохранит CSV:

```text
D:/Documents/GitHub/YAPSOD/results_mushroom_cnn_r_efficientnet/batch_predictions.csv
```


In [7]:

library(reticulate)
library(ggplot2)

if (Sys.getenv("RETICULATE_PYTHON") == "") {
  Sys.setenv(RETICULATE_PYTHON = "C:/Users/geezix/.cache/codex-runtimes/codex-primary-runtime/dependencies/python/python.exe")
}
use_python(Sys.getenv("RETICULATE_PYTHON"), required = TRUE)

predict_code <- r"---(
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from pathlib import Path

_ENSEMBLE_CACHE = {}


def _load_ensemble(results_dir):
    results_dir = Path(results_dir)
    key = str(results_dir.resolve())
    if key in _ENSEMBLE_CACHE:
        return _ENSEMBLE_CACHE[key]

    class_names = pd.read_csv(results_dir / "class_names.csv")["class"].tolist()
    feature_model = keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3),
        pooling="avg"
    )
    head_paths = sorted(results_dir.glob("mushroom_efficientnet_head_seed_*.keras"))
    if not head_paths:
        raise FileNotFoundError("Не найдены ensemble head-модели: mushroom_efficientnet_head_seed_*.keras")
    heads = [keras.models.load_model(path, compile=False) for path in head_paths]
    _ENSEMBLE_CACHE[key] = (class_names, feature_model, heads, head_paths)
    return _ENSEMBLE_CACHE[key]


def predict_mushroom_ensemble(image_path, results_dir="results_mushroom_cnn_r_efficientnet", min_confidence=0.65, min_margin=0.15):
    class_names, feature_model, heads, head_paths = _load_ensemble(results_dir)

    image = tf.io.read_file(str(image_path))
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, (224, 224))
    features = feature_model.predict(tf.expand_dims(image, 0), verbose=0)

    probabilities = np.mean([head.predict(features, verbose=0)[0] for head in heads], axis=0)
    order = np.argsort(probabilities)[::-1]
    top1 = order[0]
    top2 = order[1]
    margin = float(probabilities[top1] - probabilities[top2])
    confidence = float(probabilities[top1])

    result = pd.DataFrame({
        "class": [class_names[i] for i in order],
        "probability": [float(probabilities[i]) for i in order]
    })
    result["probability_percent"] = (result["probability"] * 100).round(2)
    return {
        "classes": [str(class_names[i]) for i in order],
        "probabilities": [float(probabilities[i]) for i in order],
        "probability_percent": [round(float(probabilities[i]) * 100, 2) for i in order],
        "top1": str(class_names[top1]),
        "top2": str(class_names[top2]),
        "confidence": confidence,
        "margin": margin,
        "models": len(heads),
        "warning": bool(confidence < min_confidence or margin < min_margin)
    }
)---"
py_run_string(predict_code)


single_image_path <- "D:/Documents/GitHub/YAPSOD/Agaricus/ttttt.jpg"

if (file.exists(single_image_path)) {
  prediction_result <- py$predict_mushroom_ensemble(
    normalizePath(single_image_path, winslash = "/", mustWork = TRUE)
  )
  prediction <- data.frame(
    class = unlist(prediction_result$classes),
    probability = as.numeric(unlist(prediction_result$probabilities)),
    probability_percent = as.numeric(unlist(prediction_result$probability_percent)),
    stringsAsFactors = FALSE
  )

  print(prediction)

  cat("\nTop-1:", prediction_result$top1, "\n")
  cat("Top-2:", prediction_result$top2, "\n")
  cat("Confidence:", round(prediction_result$confidence * 100, 2), "%\n")
  cat("Margin:", round(prediction_result$margin * 100, 2), "п.п.\n")
  cat("Models in ensemble:", prediction_result$models, "\n")

  if (isTRUE(prediction_result$warning)) {
    cat("\nПредупреждение: модель не уверена. Нужно смотреть top-2/top-3, а не только первый класс.\n")
  }

  prediction_plot <- prediction |>
    head(5) |>
    ggplot(aes(x = reorder(class, probability_percent), y = probability_percent)) +
    geom_col(fill = "#3B6EA8") +
    geom_text(aes(label = paste0(probability_percent, "%")), hjust = -0.1, size = 4) +
    coord_flip() +
    scale_y_continuous(limits = c(0, 100), expand = expansion(mult = c(0, 0.08))) +
    labs(title = "Top-5 предсказаний ансамбля", x = "Класс", y = "Вероятность, %") +
    theme_minimal(base_size = 13)

  print(prediction_plot)
} else {
  cat("Файл для одиночной проверки не найден:", single_image_path, "\n")
}

# Массовая проверка изображений.
# Загружай файлы сюда:
batch_dir <- "D:/Documents/GitHub/YAPSOD/predict_images"
results_dir <- "D:/Documents/GitHub/YAPSOD/results_mushroom_cnn_r_efficientnet"
dir.create(batch_dir, recursive = TRUE, showWarnings = FALSE)

batch_files <- list.files(
  batch_dir,
  pattern = "\\.(jpg|jpeg|png|bmp|webp)$",
  full.names = TRUE,
  ignore.case = TRUE
)

if (length(batch_files) == 0) {
  cat("\nПапка для массовой проверки пустая.\n")
  cat("Загрузи изображения сюда:\n", batch_dir, "\n", sep = "")
} else {
  predict_one_file <- function(path) {
    result <- py$predict_mushroom_ensemble(
      normalizePath(path, winslash = "/", mustWork = TRUE)
    )
    data.frame(
      file = basename(path),
      path = normalizePath(path, winslash = "/", mustWork = TRUE),
      top1 = result$top1,
      top1_probability = round(result$confidence * 100, 2),
      top2 = result$top2,
      margin_pp = round(result$margin * 100, 2),
      uncertain = isTRUE(result$warning),
      stringsAsFactors = FALSE
    )
  }

  batch_predictions <- do.call(rbind, lapply(batch_files, predict_one_file))
  batch_predictions <- batch_predictions[order(-batch_predictions$top1_probability), ]

  output_csv <- file.path(results_dir, "batch_predictions.csv")
  write.csv(batch_predictions, output_csv, row.names = FALSE, fileEncoding = "UTF-8")

  print(batch_predictions)
  cat("\nCSV с результатами сохранен:\n", output_csv, "\n", sep = "")

  batch_plot <- batch_predictions |>
    ggplot(aes(x = reorder(file, top1_probability), y = top1_probability, fill = top1)) +
    geom_col() +
    geom_text(aes(label = paste0(top1, " ", top1_probability, "%")), hjust = -0.05, size = 3.5) +
    coord_flip() +
    scale_y_continuous(limits = c(0, 100), expand = expansion(mult = c(0, 0.18))) +
    labs(title = "Массовая проверка изображений", x = "Файл", y = "Вероятность top-1, %", fill = "Класс") +
    theme_minimal(base_size = 13)

  print(batch_plot)
  ggsave(file.path(results_dir, "08_batch_predictions.png"), batch_plot, width = 9, height = max(4, min(12, 0.35 * nrow(batch_predictions) + 2)), dpi = 300)
}


Файл для одиночной проверки не найден: D:/Documents/GitHub/YAPSOD/Agaricus/ttttt.jpg 

Папка для массовой проверки пустая.
Загрузи изображения сюда:
D:/Documents/GitHub/YAPSOD/predict_images
